<a target="_blank" href="https://colab.research.google.com/github/AI4Finance-Foundation/FinRL-Tutorials/blob/master/1-Introduction/China_A_share_market_tushare.ipynb">
  <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/>
</a>

## Quantitative trading in China A stock market with FinRL

[LZ]: Change current directory to `FinRL-Meta`

In [1]:
%cd /Users/raymondl/Projects/FinRL-Meta

/Users/raymondl/Projects/FinRL-Meta


## Import Modules

In [2]:
%load_ext autoreload
%autoreload 2
%matplotlib inline

In [3]:
import warnings

warnings.filterwarnings("ignore")

import pandas as pd 
from IPython import display

display.set_matplotlib_formats("svg")

from meta import config 
from meta.data_processor import DataProcessor 
# from main import check_and_make_directories 
from meta.data_processors.tushare import Tushare, ReturnPlotter 
from meta.env_stock_trading.env_stocktrading_China_A_shares import StockTradingEnv 
from agents.stablebaselines3_models import DRLAgent 
import os 
from typing import List 
from argparse import ArgumentParser 
from meta import config 
from meta.config_tickers import DOW_30_TICKER 
from meta.config import ( DATA_SAVE_DIR, TRAINED_MODEL_DIR, TENSORBOARD_LOG_DIR, RESULTS_DIR, INDICATORS, TRAIN_START_DATE, TRAIN_END_DATE, TEST_START_DATE, TEST_END_DATE, TRADE_START_DATE, TRADE_END_DATE, ERL_PARAMS, RLlib_PARAMS, SAC_PARAMS, ALPACA_API_KEY, ALPACA_API_SECRET, ALPACA_API_BASE_URL, )

import pyfolio
from pyfolio import timeseries

pd.options.display.max_columns = None

print("ALL Modules have been imported!")

ALL Modules have been imported!


## Create Folders

In [4]:
# import os
from pathlib import Path

def check_and_make_directories(x:list[str]) -> None:
    tmp_dir = '/Users/raymondl/Projects/tmp'
    for directory in x:
        path = Path(tmp_dir)/directory
        path.mkdir(parents=True, exist_ok=True)

check_and_make_directories([DATA_SAVE_DIR, TRAINED_MODEL_DIR, TENSORBOARD_LOG_DIR, RESULTS_DIR])

## Download data, cleaning and feature engineering

In [5]:
ticker_list = ['600000.SH', '600009.SH', '600016.SH', '600028.SH', '600030.SH', '600031.SH', '600036.SH', '600050.SH', '600104.SH', '600196.SH', '600276.SH', '600309.SH', '600519.SH', '600547.SH', '600570.SH']

TRAIN_START_DATE = '2015-01-01' 
TRAIN_END_DATE= '2019-08-01' 
TRADE_START_DATE = '2019-08-01' 
TRADE_END_DATE = '2020-01-03'

TIME_INTERVAL = "1d" 
kwargs = {} 
kwargs['token'] = '27080ec403c0218f96f388bca1b1d85329d563c91a43672239619ef5' 
p = DataProcessor(data_source='tushare', start_date=TRAIN_START_DATE, end_date=TRADE_END_DATE, time_interval=TIME_INTERVAL, **kwargs)

tushare successfully connected


### Download and Clean

In [6]:
p.download_data(ticker_list=ticker_list)
p.clean_data()
p.fillna()

100%|██████████| 15/15 [00:32<00:00,  2.15s/it]

Download complete! Dataset saved to ~/Projects/tmp/tsdata/dataset.csv. 
Shape of DataFrame: (17960, 8)
Shape of DataFrame:  (18315, 8)


In [7]:
p.dataframe.head()

,tic,time,open,high,low,close,adjusted_close,volume
0,600000.SH,2015-01-05,15.88,16.25,15.56,16.07,16.07,5135687.09
1,600009.SH,2015-01-05,19.82,20.91,19.82,20.53,20.53,371485.54
2,600016.SH,2015-01-05,10.87,10.96,10.50,10.78,10.78,9138873.70
3,600028.SH,2015-01-05,6.59,7.14,6.45,7.14,7.14,11864996.45
4,600030.SH,2015-01-05,33.90,35.25,33.01,34.66,34.66,6986272.15


### Add technical indicator

In [8]:
p.add_technical_indicator(config.INDICATORS) 
p.fillna()

#print(f"p.dataframe: {p.dataframe}")

tech_indicator_list:  ['macd', 'boll_ub', 'boll_lb', 'rsi_30', 'cci_30', 'dx_30', 'close_30_sma', 'close_60_sma']
indicator:  macd
indicator:  boll_ub
indicator:  boll_lb
indicator:  rsi_30
indicator:  cci_30
indicator:  dx_30
indicator:  close_30_sma
indicator:  close_60_sma
Succesfully add technical indicators
Shape of DataFrame:  (18270, 17)


In [9]:
p.dataframe.shape, p.dataframe.index.nunique()

((18270, 17), 18270)

In [10]:
p.dataframe.head()

,tic,time,index,open,high,low,close,adjusted_close,volume,macd,boll_ub,boll_lb,rsi_30,cci_30,dx_30,close_30_sma,close_60_sma
0,600000.SH,2015-01-08,45,15.87,15.88,15.20,15.25,15.25,3306271.72,-0.032571,16.617911,15.012089,6.058641,-125.593009,20.602022,15.8150,15.8150
1,600009.SH,2015-01-08,46,20.18,20.18,19.73,20.00,20.00,198117.45,-0.016008,20.663897,19.736103,12.828915,-90.842491,100.000000,20.2000,20.2000
2,600016.SH,2015-01-08,47,10.61,10.66,10.09,10.20,10.20,4851684.17,-0.018247,10.957604,9.997396,11.862558,-99.887006,100.000000,10.4775,10.4775
3,600028.SH,2015-01-08,48,7.09,7.41,6.83,6.85,6.85,8190902.35,-0.008227,7.342000,6.743000,27.409248,36.578171,64.602490,7.0425,7.0425
4,600030.SH,2015-01-08,49,36.40,36.70,34.68,35.25,35.25,6376268.69,0.032910,36.576444,33.808556,61.517448,47.947020,100.000000,35.1925,35.1925


## Split training dataset

In [11]:
train = p.data_split(p.dataframe, TRAIN_START_DATE, TRAIN_END_DATE) 

print(f"len(train.index.levels[1].unique()): {len(train.index.levels[1].unique())}")

len(train.index.levels[1].unique()): 15


In [12]:
print(f"train.head()    :\n {train.head()}")

train.head()    :
                       index   open   high    low  close  adjusted_close  \
time       tic                                                            
2015-01-08 600000.SH     45  15.87  15.88  15.20  15.25           15.25   
           600009.SH     46  20.18  20.18  19.73  20.00           20.00   
           600016.SH     47  10.61  10.66  10.09  10.20           10.20   
           600028.SH     48   7.09   7.41   6.83   6.85            6.85   
           600030.SH     49  36.40  36.70  34.68  35.25           35.25   

                          volume      macd    boll_ub    boll_lb     rsi_30  \
time       tic                                                                
2015-01-08 600000.SH  3306271.72 -0.032571  16.617911  15.012089   6.058641   
           600009.SH   198117.45 -0.016008  20.663897  19.736103  12.828915   
           600016.SH  4851684.17 -0.018247  10.957604   9.997396  11.862558   
           600028.SH  8190902.35 -0.008227   7.342000   6.74

In [13]:
print(f"train.shape: {train.shape}")

train.shape: (16710, 15)


In [14]:
train.index.unique()

MultiIndex([('2015-01-08', '600000.SH'),
            ('2015-01-08', '600009.SH'),
            ('2015-01-08', '600016.SH'),
            ('2015-01-08', '600028.SH'),
            ('2015-01-08', '600030.SH'),
            ('2015-01-08', '600031.SH'),
            ('2015-01-08', '600036.SH'),
            ('2015-01-08', '600050.SH'),
            ('2015-01-08', '600104.SH'),
            ('2015-01-08', '600196.SH'),
            ...
            ('2019-08-01', '600031.SH'),
            ('2019-08-01', '600036.SH'),
            ('2019-08-01', '600050.SH'),
            ('2019-08-01', '600104.SH'),
            ('2019-08-01', '600196.SH'),
            ('2019-08-01', '600276.SH'),
            ('2019-08-01', '600309.SH'),
            ('2019-08-01', '600519.SH'),
            ('2019-08-01', '600547.SH'),
            ('2019-08-01', '600570.SH')],
           names=['time', 'tic'], length=16710)

Why `state_space` is length of `config.INDICATOR` plus 2 and one more extra?
initial_func[1] + close_price[s] + current_hold[s] + indicator[s*c]

In [15]:
stock_dimension = len(train.index.levels[1].unique()) 
state_space = stock_dimension * (len(config.INDICATORS) + 2) + 1 

print(f"Stock Dimension: {stock_dimension}, State Space: {state_space}")

Stock Dimension: 15, State Space: 151


## Train

In [16]:
env_kwargs = { 
    "stock_dim": stock_dimension, 
    "hmax": 1000, 
    "initial_amount": 1_000_000, 
    "buy_cost_pct": 6.87e-5,
    "sell_cost_pct": 1.0687e-3, 
    "reward_scaling": 1e-4, 
    "state_space": state_space, 
    "action_dim": stock_dimension, 
    "tech_indicator_list": config.INDICATORS, 
    "print_verbosity": 1, 
    "initial_buy": True, 
    "hundred_each_trade": True 
    }

e_train_gym = StockTradingEnv(df=train, **env_kwargs)

In [17]:
env_train, _ = e_train_gym.get_sb_env() 

print(f"print(type(env_train)): {print(type(env_train))}")

<class 'stable_baselines3.common.vec_env.vec_normalize.VecNormalize'>
print(type(env_train)): None


In [ ]:
env_train

### DDPG

In [18]:
agent = DRLAgent(env=env_train) 
DDPG_PARAMS = { 
    "batch_size": 256, 
    "buffer_size": 50000, 
    "learning_rate": 1e-4, 
    "action_noise": "normal", 
    } 
POLICY_KWARGS = dict(net_arch=dict(pi=[64, 64], qf=[400, 300])) 

model_ddpg = agent.get_model("ddpg", model_kwargs=DDPG_PARAMS, policy_kwargs=POLICY_KWARGS)
trained_ddpg = agent.train_model(model=model_ddpg, tb_log_name='ddpg', total_timesteps=15000)

{'batch_size': 256, 'buffer_size': 50000, 'learning_rate': 0.0001, 'action_noise': NormalActionNoise(mu=[0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0. 0.], sigma=[0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1])}
Using cpu device
Logging to ./tensorboard_log/ddpg/ddpg_45
Episode: 2 end.
day: 1113, episode: 2
begin_total_asset: 1000000.00
end_total_asset: 1714255.64
total_reward: 714255.64
total_cost: 15062.96
total_trades: 4464
Sharpe: 0.562
Episode: 3 end.
day: 1113, episode: 3
begin_total_asset: 1000000.00
end_total_asset: 1543666.13
total_reward: 543666.13
total_cost: 8055.53
total_trades: 4831
Sharpe: 0.479
Episode: 4 end.
day: 1113, episode: 4
begin_total_asset: 1000000.00
end_total_asset: 2248600.71
total_reward: 1248600.71
total_cost: 21683.29
total_trades: 7848
Sharpe: 0.740
Episode: 5 end.
day: 1113, episode: 5
begin_total_asset: 1000000.00
end_total_asset: 1627688.64
total_reward: 627688.64
total_cost: 18838.09
total_trades: 8142
Sharpe: 0.530
-----------------------

### A2C

agent = DRLAgent(env=env_train) 
model_a2c = agent.get_model("a2c")

trained_a2c = agent.train_model(model=model_a2c, tb_log_name='a2c', total_timesteps=50000)

### SAC

agent = DRLAgent(env=env_train) 
model_sac = agent.get_model("sac")

trained_sac = agent.train_model(model=model_sac, tb_log_name='sac', total_timesteps=80000)

## Trade

In [19]:
trade = p.data_split(p.dataframe, TRADE_START_DATE, TRADE_END_DATE) 

env_kwargs = {
    "stock_dim": stock_dimension, 
    "hmax": 1000, 
    "initial_amount": 1000000, 
    "buy_cost_pct": 6.87e-5, 
    "sell_cost_pct": 1.0687e-3, 
    "reward_scaling": 1e-4, 
    "state_space": state_space, 
    "action_dim": stock_dimension, 
    "tech_indicator_list": config.INDICATORS, 
    "print_verbosity": 1, "initial_buy": False, 
    "hundred_each_trade": True 
    } 

e_trade_gym = StockTradingEnv(df=trade, **env_kwargs)

In [20]:
df_account_value, df_actions = DRLAgent.DRL_prediction(model=trained_ddpg, environment=e_trade_gym)

Episode: 1 end.
day: 104, episode: 1
begin_total_asset: 1000000.00
end_total_asset: 1082607.51
total_reward: 82607.51
total_cost: 5707.87
total_trades: 898
Sharpe: 1.276


temp_env, _ = e_trade_gym.get_sb_env()
temp_df = temp_env.env_method(method_name="save_asset_memory")

In [21]:
df_actions.to_csv("action.csv", index=False) 
print(f"df_actions: {df_actions}")

df_actions:             600000.SH  600009.SH  600016.SH  600028.SH  600030.SH  600031.SH  \
2019-08-02     2457.0     3967.0     2279.0        0.0        0.0    10451.0   
2019-08-05      214.0      165.0       -4.0      558.0      279.0      228.0   
2019-08-06      283.0       45.0     -790.0      412.0      349.0      413.0   
2019-08-07       25.0        3.0     -862.0       33.0       80.0       81.0   
2019-08-08       37.0        9.0     -623.0        4.0       79.0       82.0   
...               ...        ...        ...        ...        ...        ...   
2019-12-27        0.0        0.0        0.0       45.0        0.0        0.0   
2019-12-30       84.0        0.0        0.0      161.0        0.0        0.0   
2019-12-31        0.0        0.0        0.0        0.0        0.0        0.0   
2020-01-02        0.0        0.0        0.0        0.0        0.0        0.0   
2020-01-03       13.0        0.0        0.0     -145.0        0.0        0.0   

            600036.SH  6000

In [ ]:
df_action.head()


In [22]:
df_actions.plot()


<Axes: >

## Backtest

### matplotlib inline

In [23]:
df_account_value =(df_account_value[["date", "total_asset"]]
                   .rename(columns={"total_asset": "account_value"})
                   .set_index('date')
                   .sort_index()
                )
        

In [ ]:
trade.info()

In [ ]:
trade.head()

In [ ]:
trade['time'] = pd.to_datetime(trade['time'])

In [ ]:
trade.dtypes

In [ ]:
trade.time.is_monotonic_increasing

In [ ]:
trade.time.head()

In [24]:

plotter = ReturnPlotter(df_account_value, trade, TRADE_START_DATE, TRADE_END_DATE)
plotter.plot()

In [25]:
# ticket: SSE 50：000016
plotter.plot("000016.SZ")

### CSI 300

In [ ]:
baseline_df = plotter.get_baseline("399300.SZ")
baseline_df = baseline_df.rename(columns={'dt': 'date'})

In [ ]:
daily_return = plotter.get_return(df_account_value)
daily_return_base = plotter.get_return(baseline_df, value_col_name="close")

perf_func = timeseries.perf_stats 
perf_stats_all = perf_func(returns=daily_return, factor_returns=daily_return_base, positions=None, transactions=None, turnover_denom="AGB")
print("==============DRL Strategy Stats===========")
print(f"perf_stats_all: {perf_stats_all}")

In [ ]:
daily_return = plotter.get_return(df_account_value)
daily_return_base = plotter.get_return(baseline_df, value_col_name="close")

perf_func = timeseries.perf_stats
perf_stats_all = perf_func(returns=daily_return_base, factor_returns=daily_return_base, positions=None, transactions=None, turnover_denom="AGB")

print("==============Baseline Strategy Stats===========")

print(f"perf_stats_all: {perf_stats_all}")